# Feature Extraction

Extracts MediaPipe FaceMesh landmark coordinates from images and exports them as CSV.

**Inputs:** `data/raw/YawDD/` and `data/raw/ISDDS/` (see README for dataset setup)  
**Outputs:** `data/facial_drowsiness_data.csv`, `data/isdds_data.csv`

In [ ]:
import os, sys

# Root of the repo — works locally and in Colab (after cloning the repo)
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(base_dir, "modules"))
print("base_dir:", base_dir)

## Install & import libraries

In [ ]:
%%capture
!pip install mediapipe

In [ ]:
from glob import glob
import random

train_images = sorted(glob(os.path.join(base_dir, "data/raw/YawDD/*/*.*p*")))
test_images  = sorted(glob(os.path.join(base_dir, "data/raw/ISDDS/*/*.*p*")))
print(f"Train images : {len(train_images)}")
print(f"Test images  : {len(test_images)}")

In [ ]:
from dfe import DrowsinessFeatureExtraction
import cv2
from datetime import datetime

## Feature extraction function

In [ ]:
def ekstrak_fitur(image_list, augment = False, normalize = False, n_visualize = 5):
  """Menerima list directory gambar, mengembalikan list dari list koordinat wajah"""
  random.shuffle(image_list)
  coords = []
  wasted = []

  for i, im in enumerate(image_list):
    label = im.split("/")[-2].lower()
    label = 1 if label == 'yawn' else 0 if label == 'no_yawn' else -1
    if label == -1:
      print("ERROR! Unrecognised directory")
      continue
    data = [label]

    im1 = cv2.imread(im)
    im1 = cv2.cvtColor(im1, cv2.COLOR_BGR2RGB)
    im1_extraction = DrowsinessFeatureExtraction(im1)

    if im1_extraction.detected:
        data.append(im1_extraction.coordinates)
        coords.append(data)
        if i < n_visualize:
            print(im)
            im1_extraction.compare()
    else:
        wasted.append(im)
        print(f"Skipping image: {wasted[len(wasted)-1]}, current wasted image: {len(wasted)}")
        log_path = os.path.join(base_dir, "logs/log.txt")
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, "a") as f:
            f.write(f"{str(datetime.now())[:-7]} | Skipping image: {wasted[len(wasted)-1]}\n")
        continue

    if augment:
      data2 = [label]
      im2 = cv2.flip(im1, 1)
      im2_extraction = DrowsinessFeatureExtraction(im2)
      if im2_extraction.detected:
        data2.append(im2_extraction.coordinates)
        coords.append(data2)

  return coords

In [ ]:
train_coords = ekstrak_fitur(train_images, augment = True)

In [ ]:
test_coords = ekstrak_fitur(test_images)

## Export to CSV

In [ ]:
import pandas as pd
def export_coordinates(coords):
  """Mengubah bentuk hasil keluaran fungsi ekstrak_fitur menjadi Pandas DataFrame"""
  datas = []
  for data in coords:
    tmp = []
    tmp.append(data[0])
    for coord in data[1]:
      for d in coord:
        tmp.append(d)
      datas.append(tmp)

  columns = ['class']
  for i in range(468):  
    columns.append(f"x{i+1}")
    columns.append(f"y{i+1}")
    columns.append(f"z{i+1}")
  
  df = pd.DataFrame(datas, columns = columns)
  return df

In [ ]:
df_train = export_coordinates(train_coords)
df_test  = export_coordinates(test_coords)

In [ ]:
os.makedirs(os.path.join(base_dir, "data"), exist_ok=True)
df_train.to_csv(os.path.join(base_dir, "data/facial_drowsiness_data.csv"))
df_test.to_csv(os.path.join(base_dir,  "data/isdds_data.csv"))
print("Saved:", os.path.join(base_dir, "data/facial_drowsiness_data.csv"))
print("Saved:", os.path.join(base_dir, "data/isdds_data.csv"))